# CoinCell — Model Training (Kaggle GPU)

**Congressional App Challenge** · Trains DualViewNet + CoinCellNet

Before running:
1. **Settings → GPU** ON
2. **Settings → Internet** ON
3. **Add-ons → Secrets → `HF_TOKEN`** (Hugging Face write token)

Only uses `/kaggle/working/` — does not touch your other Kaggle notebooks/datasets.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

WORK = Path('/kaggle/working')
SRC = WORK / 'coincell-src'
REPO = 'https://github.com/arjunkshah12345-hash/coincell.git'

if not SRC.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(SRC)], check=True)
else:
    subprocess.run(['git', '-C', str(SRC), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(SRC))
print('Source:', SRC)
print('GPU available:', __import__('torch').cuda.is_available())

In [ ]:
import torch
from coincell.classifier import build_dataset, train_models, save_models
from coincell.evaluate import evaluate_on_synthetic

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on {DEVICE}')

# Override build for larger Kaggle GPU run
from coincell import classifier
orig_build = classifier.build_dataset

def kaggle_build(n_per_class=400, size=224, dual=True):
    return orig_build(n_per_class=n_per_class, size=size, dual=dual)

classifier.build_dataset = kaggle_build

EPOCHS = 20
single, dual = train_models(epochs=EPOCHS, batch_size=64, device=DEVICE)

weights_path = WORK / 'coincell.pt'
save_models(single, dual, weights_path)
print(f'Saved → {weights_path} ({weights_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
# Evaluate ensemble (CV + Kaggle-trained CNN)
import os
os.environ['COINCELL_WEIGHTS'] = str(WORK / 'coincell.pt')

# Clear cached engine so it reloads Kaggle weights
import coincell.inference as inf
inf._engine = None

metrics = evaluate_on_synthetic(n=80)
metrics_path = WORK / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
# Upload to Hugging Face Hub (inference app pulls from here)
from huggingface_hub import HfApi, create_repo
from kaggle_secrets import UserSecretsClient

HF_REPO = 'arjunkshah12345-hash/coincell-weights'
token = UserSecretsClient().get_secret('HF_TOKEN')
api = HfApi(token=token)

create_repo(HF_REPO, repo_type='model', exist_ok=True)
api.upload_file(str(weights_path), 'coincell.pt', repo_id=HF_REPO, repo_type='model',
                commit_message=f'CoinCell Kaggle train — {EPOCHS} epochs GPU')
api.upload_file(str(metrics_path), 'metrics.json', repo_id=HF_REPO, repo_type='model',
                commit_message='Evaluation metrics')

print(f'✓ Weights live: https://huggingface.co/{HF_REPO}')
print('Next: deploy HF Space → python3 scripts/deploy_space.py')